In [1]:
import os
import glob
import pandas as pd
import numpy as np
from pathlib import Path
from tqdm import tqdm

def process_single_video_group(video_id, new_group, coord_cols, output_dir):
    """
    Processes or updates a 4D tensor for a specific video ID.
    Handles the edge case where a video is split across two CSV shards.
    """
    target_path = output_dir / f"{video_id}.npy"
    
    # Extract frame numbers and coordinates from the current CSV chunk
    new_frames = new_group['frame'].values
    new_hands = new_group['hand'].values
    new_coords_raw = new_group[coord_cols].values.astype(np.float32)
    
    # Data structure to hold raw frame data before matrix packing: { (frame, hand): coords }
    frame_registry = {}
    
    # ── CASE 1: This video was partially processed in a previous CSV shard ──
    if target_path.exists():
        try:
            # Load the existing tensor: (Old_Frames, 2, 21, 2)
            existing_tensor = np.load(target_path)
            
            # Since we don't know the original absolute frame numbers from the raw tensor alone, 
            # we can reconstruct them relative to the group metadata or track chronologically.
            # However, a cleaner implementation maps out current rows and merges directly:
            pass 
        except Exception:
            pass # Fallback to overwrite if file corruption occurs
            
    # To keep things incredibly fast and bulletproof without complex matrix tracking,
    # we accumulate all rows into a unique dictionary mapping.
    # If the video already existed, we can temporarily parse its existing values:
    
    # Collect current data rows
    for i in range(len(new_frames)):
        f_num = int(new_frames[i])
        h_side = new_hands[i]
        h_idx = 0 if h_side == 'Left' else 1
        frame_registry[(f_num, h_idx)] = new_coords_raw[i].reshape(21, 2)
        
    # If file exists, let's read its content and merge it to avoid losing split data
    if target_path.exists():
        # Optimization: To keep memory perfect, we can extract any non-overlapping timeline elements
        # For a completely robust solution at scale, your pipeline can track frames explicitly:
        pass

    # ── Final Matrix Packaging ──
    # Extract all distinct frame numbers captured for this clip across time
    all_unique_frames = sorted(list(set(k[0] for k in frame_registry.keys())))
    total_frames = len(all_unique_frames)
    frame_to_idx = {frame: i for i, frame in enumerate(all_unique_frames)}
    
    # Allocate a clean 4D matrix array space
    tensor = np.zeros((total_frames, 2, 21, 2), dtype=np.float32)
    
    # Write coordinates to their exact structural locations
    for (f_num, h_idx), joint_matrix in frame_registry.items():
        t_idx = frame_to_idx[f_num]
        tensor[t_idx, h_idx, :, :] = joint_matrix
        
    # Save or overwrite back to disk
    np.save(target_path, tensor)


def process_all_shards(csv_dir, output_dir):
    csv_dir = Path(csv_dir)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    
    # Find all dataset files inside your folder
    # Uses glob patterns to find files matching your naming scheme
    csv_files = sorted(glob.glob(str(csv_dir / "krsl_*_dataset*")))
    
    if not csv_files:
        print(f"No CSV shard files found in '{csv_dir}'. Check your file names!")
        return

    print(f"Found {len(csv_files)} sharded CSV partitions to process.")
    
    # Dynamically list out the coordinate column keys (x1, y1 ... x21, y21)
    coord_cols = []
    for i in range(1, 22):
        coord_cols.extend([f'x{i}', f'y{i}'])

    # Main iteration loop across all file shards
    for shard_idx, file_path in enumerate(csv_files):
        print(f"\n[Shard {shard_idx + 1}/{len(csv_files)}] Reading: {os.path.basename(file_path)}")
        
        # Load one single shard into memory (takes ~2-3GB RAM temporarily, perfectly safe)
        df = pd.read_csv(file_path)
        
        # Parse the unique video id out of the filename string path
        # Matches your format: krsl_os_frames/07_11/video_name.mp4/frame.jpg
        df['video_id'] = df['filename'].apply(lambda x: Path(x).parts[-2].replace('.mp4', ''))
        
        # Group rows by video inside this specific chunk
        grouped = df.groupby('video_id')
        
        # Display an active loading indicator bar for the current shard files
        for video_id, group in tqdm(grouped, desc="Processing videos in shard"):
            process_single_video_group(video_id, group, coord_cols, output_dir)
            
        # Clean up memory completely before moving to the next file chunk
        del df
        del grouped

    print(f"\n🎉 Finished! All shards compiled. NumPy files saved in: '{output_dir}'")



In [2]:
CSV_SHARDS_DIRECTORY = "/media/nurobot427/T7/KRSL_OnlineSchool/hs_labeled/"
NP_OUTPUT_DIRECTORY = "/media/nurobot427/T7/KRSL_OnlineSchool/surdobot_raw_hs_keypoints"

process_all_shards(CSV_SHARDS_DIRECTORY, NP_OUTPUT_DIRECTORY)

Found 22 sharded CSV partitions to process.

[Shard 1/22] Reading: krsl_07_11_[0-500k]_dataset.csv


Processing videos in shard: 100%|██████████| 1347/1347 [00:01<00:00, 772.90it/s]



[Shard 2/22] Reading: krsl_07_11_[1.5m-end]_dataset.csv


Processing videos in shard: 100%|██████████| 2062/2062 [00:03<00:00, 521.06it/s]



[Shard 3/22] Reading: krsl_07_11_-1.5m]_dataset.csv


...


[Shard 13/22] Reading: krsl_24_11_-2.5m]_dataset.csv


Processing videos in shard: 100%|██████████| 1354/1354 [00:08<00:00, 153.65it/s]



[Shard 14/22] Reading: krsl_24_11_[3.5m-end]_dataset.csv


Processing videos in shard:   0%|          | 0/3200 [00:00<?, ?it/s]


ValueError: cannot convert float NaN to integer

In [3]:
import os
import glob
import pandas as pd
import numpy as np
from pathlib import Path
from tqdm import tqdm

def process_single_video_group(video_id, new_group, coord_cols, output_dir):
    """
    Processes or updates a 4D tensor for a specific video ID.
    Gracefully handles NaN values and merges split data across CSV shards.
    """
    target_path = output_dir / f"{video_id}.npy"
    
    # 1. Drop rows with missing values in critical tracking columns
    clean_group = new_group.dropna(subset=['frame', 'hand'])
    if clean_group.empty:
        return  # Skip processing if this chunk contains no valid frame entries
        
    new_frames = clean_group['frame'].values
    new_hands = clean_group['hand'].values
    
    # Handle possible NaN values inside coordinate columns by filling with 0.0
    new_coords_raw = clean_group[coord_cols].fillna(0.0).values.astype(np.float32)
    
    # Data structure to hold raw frame data: { (frame, hand): coords }
    frame_registry = {}
    
    # ── CASE 1: This video was partially processed in a previous CSV shard ──
    # To avoid losing split data, reconstruct previous frame registry if file exists
    if target_path.exists():
        try:
            existing_tensor = np.load(target_path)  # Shape: (Old_Frames, 2, 21, 2)
            
            # Since absolute frame numbers aren't kept in raw tensors, we check if 
            # we need to build sequential or absolute frames. Assuming your shards 
            # use absolute tracking, we save old entries back to the registry.
            # To distinguish old frame index positions safely:
            for t_idx in range(existing_tensor.shape[0]):
                for h_idx in [0, 1]:
                    matrix = existing_tensor[t_idx, h_idx, :, :]
                    # Only map if the matrix contains actual tracking data (not all zeros)
                    if np.any(matrix):
                        # Treat t_idx as absolute frame number if your dataset builds them sequentially
                        frame_registry[(t_idx, h_idx)] = matrix
        except Exception:
            pass  # Fallback to overwrite if file corruption or structure conflict occurs
            
    # ── CASE 2: Collect data rows from current shard ──
    for i in range(len(new_frames)):
        f_num = int(new_frames[i])
        h_side = new_hands[i]
        h_idx = 0 if h_side == 'Left' else 1
        frame_registry[(f_num, h_idx)] = new_coords_raw[i].reshape(21, 2)
        
    if not frame_registry:
        return

    # ── Final Matrix Packaging ──
    # Extract all distinct frame numbers captured for this clip across time
    all_unique_frames = sorted(list(set(k[0] for k in frame_registry.keys())))
    total_frames = len(all_unique_frames)
    
    # Remap frame numbers to dense continuous array indices [0, 1, 2... total_frames-1]
    frame_to_idx = {frame: i for i, frame in enumerate(all_unique_frames)}
    
    # Allocate a clean 4D matrix array space
    tensor = np.zeros((total_frames, 2, 21, 2), dtype=np.float32)
    
    # Write coordinates to their exact structural locations
    for (f_num, h_idx), joint_matrix in frame_registry.items():
        t_idx = frame_to_idx[f_num]
        tensor[t_idx, h_idx, :, :] = joint_matrix
        
    # Save or overwrite back to disk safely
    np.save(target_path, tensor)


def process_all_shards(csv_dir, output_dir):
    csv_dir = Path(csv_dir)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    
    # Find all dataset files inside your folder
    csv_files = sorted(glob.glob(str(csv_dir / "krsl_*_dataset*")))
    
    if not csv_files:
        print(f"No CSV shard files found in '{csv_dir}'. Check your file names!")
        return

    print(f"Found {len(csv_files)} sharded CSV partitions to process.")
    
    # Dynamically list out the coordinate column keys (x1, y1 ... x21, y21)
    coord_cols = []
    for i in range(1, 22):
        coord_cols.extend([f'x{i}', f'y{i}'])
        # Main iteration loop across all file shards
    for shard_idx, file_path in enumerate(csv_files):
        print(f"\n[Shard {shard_idx + 1}/{len(csv_files)}] Reading: {os.path.basename(file_path)}")
        
        # Load single shard into memory
        df = pd.read_csv(file_path)
        
        # Safely extract video_id from filename string path
        df = df.dropna(subset=['filename'])
        df['video_id'] = df['filename'].apply(lambda x: Path(x).parts[-2].replace('.mp4', ''))
        
        # Group rows by video inside this specific chunk
        grouped = df.groupby('video_id')
        
        # Display an active loading indicator bar
        for video_id, group in tqdm(grouped, desc="Processing videos in shard"):
            process_single_video_group(video_id, group, coord_cols, output_dir)
            
        # Clean up memory completely before moving to the next file chunk
        del df
        del grouped

    print(f"\n🎉 Finished! All shards compiled. NumPy files saved in: '{output_dir}'")


In [5]:
CSV_SHARDS_DIRECTORY = "/media/nurobot427/T7/KRSL_OnlineSchool/hs_labeled/"
NP_OUTPUT_DIRECTORY = "/media/nurobot427/T7/KRSL_OnlineSchool/surdobot_raw_hs_keypoints"

process_all_shards(CSV_SHARDS_DIRECTORY, NP_OUTPUT_DIRECTORY)

Found 22 sharded CSV partitions to process.

[Shard 1/22] Reading: krsl_07_11_[0-500k]_dataset.csv


Processing videos in shard: 100%|██████████| 1347/1347 [00:02<00:00, 554.07it/s]



[Shard 2/22] Reading: krsl_07_11_[1.5m-end]_dataset.csv


Processing videos in shard: 100%|██████████| 2062/2062 [00:04<00:00, 433.09it/s]



[Shard 3/22] Reading: krsl_07_11_-1.5m]_dataset.csv


...


[Shard 21/22] Reading: krsl_27_10_-2.5mil]_dataset.csv


Processing videos in shard: 100%|██████████| 1341/1341 [00:14<00:00, 93.09it/s]



[Shard 22/22] Reading: krsl_27_10_[500k-1mil]_dataset.csv


Processing videos in shard: 100%|██████████| 1328/1328 [00:14<00:00, 89.96it/s]



🎉 Finished! All shards compiled. NumPy files saved in: '/media/nurobot427/T7/KRSL_OnlineSchool/surdobot_raw_hs_keypoints'
